#### Initial Set Up

In [1]:
# Cell 1 - Initial Model Set Up
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="cuda" 
)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [2]:
# Cell 2 - Load GSM8K dataset
from datasets import load_dataset

dataset = load_dataset("openai/gsm8k", "main")
train_set = dataset["train"]
TOTAL_TRAIN_COUNT = len(train_set)
test_set = dataset["test"]
TOTAL_TEST_COUNT = len(test_set)
print(f"Total test samples: {TOTAL_TEST_COUNT}")

Total test samples: 1319


#### Experiment 1: Test Accuracy of ZERO SHOT STRUCTURED PROMPT

In [3]:
# Cell 3 - Zero Shot prompt
def zero_shot_prompt(question):
    prompt = f"""Solve the following math problems. Show step by step. Always end your answer with #### <number> where <number> is the final answer.
Q: {question}
A:"""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}  # ← must not be commented out

    output = model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=True,
        temperature=0.3,
        top_p=0.95
    )

    input_length = inputs["input_ids"].shape[1]
    generated_tokens = output[0][input_length:]  # ← only decode new tokens
    return tokenizer.decode(generated_tokens, skip_special_tokens=True)

In [4]:
# Cell 4 - test zero shot prompt
question = test_set[0]["question"]
print(question)
answer = zero_shot_prompt(question)
print(answer)

Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
To solve this problem, we need to follow these steps:

1. Calculate how many eggs Janet eats for breakfast each day.
2. Subtract the number of eggs eaten for breakfast from the total number of eggs laid each day.
3. Subtract the number of eggs used to bake muffins from the remaining eggs.
4. Calculate the number of eggs left after baking muffins.
5. Multiply the number of leftover eggs by the price per egg to find out how much money she makes.

Let's go through each step:

Step 1: Eggs eaten for breakfast
Janet eats 3 eggs for breakfast every morning.

Step 2: Remaining eggs after breakfast
Total eggs laid = 16
Eggs eaten for breakfast = 3
Remaining eggs after breakfast = Total eggs - Eggs eaten for breakfast


In [5]:
# Cell 5 -compare answer and correct answer using math-verify
from math_verify import verify, parse

def check(generated_answer, correct_answer):
    try:
       gold = parse(correct_answer, parsing_timeout=None)[0]
       pred = parse(generated_answer, parsing_timeout=None)[0]
       return bool(verify(gold, pred, timeout_seconds=None))
    except Exception as e:
        print(f"Error during verification: {e}")
        return False

In [6]:
# Cell 6 - test check function
check(answer, test_set["answer"][0])

Timeout is disabled as parsing_timeout is None or <= 0, you must provide                         the logic for timeout interuption yourself to prevent code getting stuck.
Timeout is disabled as timeout_seconds is None or <= 0, you must provide                         the logic for timeout interuption yourself to prevent code getting stuck.


True

In [7]:
# Cell 7 - Run zero shot evaluation on the test set
correct = 0
for i in range(TOTAL_TEST_COUNT):
    Q = test_set["question"][i]
    ans = zero_shot_prompt(Q)
    if check(ans, test_set["answer"][i]):
        correct += 1
print(correct)
print(TOTAL_TEST_COUNT)
print(f"Accuracy: {correct/TOTAL_TEST_COUNT:.2f}")

KeyboardInterrupt: 

#### Experiment 2: Test Accuracy of  RANDOM FEW SHOT PROMPT

In [ ]:
# Cell 8 - Few shot prompt
import random

def random_few_shot_prompt(question, k=3):
    few_shots = []
    for i in range(k):
        idx = random.randint(0, TOTAL_TRAIN_COUNT-1)
        few_shots.append(train_set[idx])
    prompt = "Solve the following math problems. Show step by step. Always end your answer with #### <number> where <number> is the final answer.\n\n"
    for fs in few_shots:    
        prompt += f"Q: {fs['question']}\nA: {fs['answer']}\n\n" 
    prompt += f"Q: {question}\nA:"
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    output = model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=True,
        temperature=0.3,
        top_p=0.95
    )
    input_length = inputs["input_ids"].shape[1]
    generated_tokens = output[0][input_length:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True)

In [ ]:
# Cell 9 - test few shot prompt
question = test_set[0]["question"]
print(question)    
answer = random_few_shot_prompt(question)
print(answer)
print(check(answer, test_set[0]["answer"]))